# Homework 5, task 3

## part 1: implementing scaled dot product attention

In [34]:
import numpy as np

def softmax(V: np.ndarray):
  return np.exp(V) / np.sum(np.exp(V))

def attention(Q: np.ndarray, K: np.ndarray, V):
  assert Q.shape[-1] == K.shape[-1]
  assert V.shape[0] == K.shape[0] #some quick sanity checks

  d = len(K)

  num = Q @ K.T
  denom = np.sqrt(d)

  return (num/denom) @ V


len_seq = 16
model_dim = 100

sequence = np.random.randn(len_seq, model_dim)
W_q = np.random.randn(model_dim, model_dim)
W_k = np.random.randn(model_dim, model_dim)

Q = W_q @ sequence.T
K = W_k @ sequence.T
V = sequence[2]


print(attention(Q, K, V).shape)


(100,)


part 2: implementing encoder-decoder seq2seq model for translation

In [35]:
from datasets import load_dataset

ds = load_dataset("bentrevett/multi30k")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.jsonl: 0.00B [00:00, ?B/s]

val.jsonl: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/29000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1014 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [36]:
print(ds['train'][0])

data = ds['train'].to_pandas()

{'en': 'Two young, White males are outside near many bushes.', 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.'}


In [37]:
import re

def tokenize(text):
  text = text.lower().strip()
  text = re.sub(r"([.,!?;])", r" \1 ", text)
  text = re.sub(r"\s+", " ", text)
  return text.split()

en_token_cache = set()
de_token_cache = set()

for i, row in data.iterrows():
  for word in row['en'].split():
    tokens = tokenize(word)
    en_token_cache.update(tokens)

  for word in row['de'].split():
    tokens = tokenize(word)
    de_token_cache.update(tokens)

print(len(en_token_cache))
print(len(de_token_cache))

10523
18792


In [38]:
en_token_dict = {token: i+3 for i, token in enumerate(en_token_cache)}
de_token_dict = {token: i+3 for i, token in enumerate(de_token_cache)}

en_token_dict.update({'<PAD>': 0, '<SOS>': 1, '<EOS>': 2})
de_token_dict.update({'<PAD>': 0, '<SOS>': 1, '<EOS>': 2})
# I want to simplify my training setup so I'm not sure I'll use all of these

for i, thing in enumerate(en_token_dict.items()):
  print(i, thing)
  if i > 10:
    break

print(de_token_dict['zwei'])

0 ("others'", 3)
1 ('rows', 4)
2 ('phones', 5)
3 ('angel', 6)
4 ('forty', 7)
5 ('accepting', 8)
6 ('hue', 9)
7 ('acquaintance', 10)
8 ('tobacco', 11)
9 ('shiny', 12)
10 ('farmlands', 13)
11 ('piggyback', 14)
1988


In [58]:
from torch import nn
import torch
import torch.nn.functional as F
import math

def softmax(V: np.ndarray):
  return np.exp(V) / np.sum(np.exp(V))

class Attention(nn.Module):
  """
  implementing scaled dot product attention in pytorch
  """
  def __init__(self, hidden_dim):
      super().__init__()
      self.hidden_dim = hidden_dim

  def forward(self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor):
      scores = torch.bmm(query, key.transpose(1, 2)) / math.sqrt(self.hidden_dim)
      attn_weights = F.softmax(scores, dim=-1)
      context = torch.bmm(attn_weights, value)

      return context, attn_weights

class Encoder(nn.Module):
  def __init__(self, lexicon_size, emb_dim, n_hidden):
    super().__init__()
    self.embedding = nn.Embedding(lexicon_size, emb_dim, padding_idx=0)

    self.rnn = nn.LSTM(input_size=emb_dim,
                      hidden_size=n_hidden,
                      num_layers=1,
                      batch_first=True)

  def forward(self, X):
    embedded = self.embedding(X)
    outputs, (hidden, cell) = self.rnn(embedded)
    return outputs, (hidden, cell)

class Decoder(nn.Module):
  def __init__(self, lexicon_size, emb_dim, n_hidden, n):
    super().__init__()
    self.embedding = nn.Embedding(lexicon_size, emb_dim, padding_idx=0)
    self.attention = Attention(n_hidden)
    self.rnn = nn.LSTM(input_size=emb_dim + n_hidden,
                      hidden_size=n_hidden,
                      num_layers=1,
                      batch_first=True)
    self.fc_out = nn.Linear(n_hidden * 2, lexicon_size)

  def forward(self, X, hidden, cell, encoder_outputs):
    embedded = self.embedding(X.unsqueeze(1))

    context, attn_weights = self.attention(hidden.squeeze(0).unsqueeze(1), encoder_outputs, encoder_outputs)
    rnn_input = torch.cat((embedded, context.unsqueeze(1)), dim=2)

    output, (hidden, cell) = self.rnn(rnn_input, (hidden, cell))
    prediction = self.fc_out(torch.cat((output.squeeze(1), context), dim=1))

    return prediction, (hidden, cell)

class Seq2Seq(nn.Module):
  def __init__(self, encoder, decoder):
    super().__init__()
    self.encoder = encoder
    self.decoder = decoder

  def forward(self, input_sequence, target_sequence):
    batch_size = input_sequence.shape[0]
    target_len = target_sequence.shape[1]
    outputs = torch.zeros(batch_size, target_len, self.decoder.fc_out.out_features).to(input_sequence.device)

    encoder_outputs, (hidden, cell) = self.encoder(input_sequence)
    decoder_input = torch.zeros(batch_size, dtype=torch.long).to(input_sequence.device)

    decoder_hidden = hidden
    decoder_cell = cell

    for t in range(target_len):
      #unrolling
      decoder_output, (decoder_hidden, decoder_cell) = self.decoder(decoder_input, decoder_hidden, decoder_cell, encoder_outputs)
      outputs[:, t, :] = decoder_output
      decoder_input = target_sequence[:, t]

    return outputs

In [40]:
def tokenize_sequence(sentence, token_dict):
  tokens = tokenize(sentence)
  return [token_dict[token] for token in tokens]


data.head()

,en,de
0,"Two young, White males are outside near many b...",Zwei junge weiße Männer sind im Freien in der ...
1,Several men in hard hats are operating a giant...,Mehrere Männer mit Schutzhelmen bedienen ein A...
2,A little girl climbing into a wooden playhouse.,Ein kleines Mädchen klettert in ein Spielhaus ...
3,A man in a blue shirt is standing on a ladder ...,Ein Mann in einem blauen Hemd steht auf einer ...
4,Two men are at the stove preparing food.,Zwei Männer stehen am Herd und bereiten Essen zu.


In [41]:
data['token_en'] = data['en'].apply(lambda x: tokenize_sequence(x, en_token_dict))
data['token_de'] = data['de'].apply(lambda x: tokenize_sequence(x, de_token_dict))

data.head()

,en,de,token_en,token_de
0,"Two young, White males are outside near many b...",Zwei junge weiße Männer sind im Freien in der ...,"[2657, 9619, 211, 4283, 3031, 73, 2597, 1121, ...","[1988, 8434, 13808, 9459, 2070, 12102, 12426, ..."
1,Several men in hard hats are operating a giant...,Mehrere Männer mit Schutzhelmen bedienen ein A...,"[939, 8810, 10269, 10316, 3933, 73, 467, 9795,...","[15048, 9459, 1578, 13018, 16483, 6042, 12163,..."
2,A little girl climbing into a wooden playhouse.,Ein kleines Mädchen klettert in ein Spielhaus ...,"[9795, 9380, 3649, 8618, 118, 9795, 8638, 3076...","[6042, 16017, 8960, 17880, 18370, 6042, 7413, ..."
3,A man in a blue shirt is standing on a ladder ...,Ein Mann in einem blauen Hemd steht auf einer ...,"[9795, 4115, 10269, 9795, 765, 8332, 7015, 151...","[6042, 15819, 18370, 3861, 17404, 6913, 12571,..."
4,Two men are at the stove preparing food.,Zwei Männer stehen am Herd und bereiten Essen zu.,"[2657, 8810, 73, 1088, 6059, 2077, 558, 5885, ...","[1988, 9459, 17651, 14780, 14292, 16932, 7819,..."


In [42]:
data.drop(columns=['en', 'de'], inplace=True)

In [43]:
data.head()

,token_en,token_de
0,"[2657, 9619, 211, 4283, 3031, 73, 2597, 1121, ...","[1988, 8434, 13808, 9459, 2070, 12102, 12426, ..."
1,"[939, 8810, 10269, 10316, 3933, 73, 467, 9795,...","[15048, 9459, 1578, 13018, 16483, 6042, 12163,..."
2,"[9795, 9380, 3649, 8618, 118, 9795, 8638, 3076...","[6042, 16017, 8960, 17880, 18370, 6042, 7413, ..."
3,"[9795, 4115, 10269, 9795, 765, 8332, 7015, 151...","[6042, 15819, 18370, 3861, 17404, 6913, 12571,..."
4,"[2657, 8810, 73, 1088, 6059, 2077, 558, 5885, ...","[1988, 9459, 17651, 14780, 14292, 16932, 7819,..."


In [44]:
def make_batch(input_sequences, target_sequences, batch_size:int):
  max_in_len = max([len(x) for x in input_sequences])
  max_target_len = max([len(x) for x in target_sequences])

  X = torch.zeros((batch_size, max_in_len), dtype=torch.long)
  Y = torch.zeros((batch_size, max_target_len), dtype=torch.long)

  for i in range(batch_size):
    current_input_sequence = input_sequences[i]
    current_target_sequence = target_sequences[i]

    padded_en_list = current_input_sequence + (max_in_len - len(current_input_sequence)) * [en_token_dict['<PAD>']]
    padded_de_list = current_target_sequence + (max_target_len - len(current_target_sequence)) * [de_token_dict['<PAD>']]

    assert len(padded_en_list) == max_in_len
    assert len(padded_de_list) == max_target_len

    X[i] = torch.tensor(padded_en_list)
    Y[i] = torch.tensor(padded_de_list)

  return X, Y

In [45]:
from sklearn.model_selection import train_test_split
from torch import nn
import torch
import torch.nn.functional as F
import math

train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

#sanity check
print(type(train_data))
print(type(test_data))

encoder = Encoder(len(en_token_dict), 256, 512)
decoder = Decoder(len(de_token_dict), 256, 512, 512)
model = Seq2Seq(encoder, decoder)

dummy_data = train_data[0:16]

print(dummy_data.head())

dummy_batch = make_batch(dummy_data['token_en'].to_list(), dummy_data['token_de'].to_list(), 16)

print(dummy_batch[0].shape)
print(dummy_batch[1].shape)

inference = model(dummy_batch[0], dummy_batch[1])

print("model inference shape:", inference.shape)

<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>
                                                token_en  \
9363   [9795, 8120, 4965, 10269, 4660, 7278, 7082, 84...   
23806  [9795, 4115, 20, 118, 9795, 340, 413, 283, 873...   
5653   [2657, 5429, 10269, 9898, 4670, 9850, 10269, 9...   
16613  [10269, 6059, 7393, 211, 7183, 73, 2919, 2433,...   
10512  [9795, 1576, 2616, 1258, 220, 1411, 6059, 5278...   

                                                token_de  
9363   [6042, 4860, 7338, 8482, 16932, 6042, 1766, 98...  
23806  [6042, 15819, 12281, 18370, 6042, 8243, 1578, ...  
5653   [1988, 8960, 18370, 14534, 9928, 18370, 3861, ...  
16613  [12102, 13725, 17651, 6626, 10743, 14780, 1958...  
10512  [11924, 10230, 13938, 1654, 2669, 9003, 17905,...  
torch.Size([16, 36])
torch.Size([16, 26])
model inference shape: torch.Size([16, 26, 18795])


###Model properly infers through the seq2seq model!

In [46]:
de_token_lookup = {}
en_token_lookup = {}

for pair in en_token_dict.items():
  en_token_lookup.update({pair[1]: pair[0]})

for pair in de_token_dict.items():
  de_token_lookup.update({pair[1]: pair[0]})

print(en_token_lookup[999])

level's


In [47]:

print(inference[-1].shape)


def print_sentence(inference, lookup):
  string = ""
  for term in inference:
    index = torch.argmax(term).item()
    string += lookup[index] + " "

  print(string)


print_sentence(inference[-1], de_token_lookup)

torch.Size([26, 18795])
tennisrock rucksäcke klammert zeremonie handpuppe kaputten allradfahrzeug tubas meißel gewartet schauspielern metallobjekt skihütte metallstab metallstab metallstab metallstab metallstab tennisrock tennisrock tennisrock tennisrock tennisrock tennisrock tennisrock tennisrock 


In [ ]:
import torch.optim as optim

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=de_token_dict['<PAD>'])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

In [ ]:
num_epochs = 5
batch_size = 32
train_en_list = train_data['token_en'].to_list()
train_de_list = train_data['token_de'].to_list()

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0

    indices = torch.randperm(len(train_en_list)).tolist()
    shuffled_en = [train_en_list[i] for i in indices]
    shuffled_de = [train_de_list[i] for i in indices]

    for i in range(0, len(shuffled_en), batch_size):
        batch_en = shuffled_en[i:i + batch_size]
        batch_de = shuffled_de[i:i + batch_size]

        if len(batch_en) == 0:
            continue

        input_tensor, target_tensor = make_batch(batch_en, batch_de, len(batch_en))

        input_tensor = input_tensor.to(device)
        target_tensor = target_tensor.to(device)

        optimizer.zero_grad()

        output = model(input_tensor, target_tensor)
        output_dim = output.shape[-1]
        output = output.reshape(-1, output_dim)
        target_tensor = target_tensor.reshape(-1)

        loss = criterion(output, target_tensor)
        loss.backward()
        optimizer.step()

        print(f"epoch completion: {i/len(shuffled_en)}")

        epoch_loss += loss.item()

    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss/len(shuffled_en):.4f}')

print("Training complete.")

(I deleted loop debug to prevent notebook clutter)

Epoch 1/5, Loss: 0.1034

Epoch 2/5, Loss: 0.0651

Epoch 3/5, Loss: 0.0386

Epoch 4/5, Loss: 0.0242

Epoch 5/5, Loss: 0.0162

In [26]:
test_en_list = test_data['token_en'].to_list()
test_de_list = test_data['token_de'].to_list()

model.eval()



with torch.no_grad():
  total_loss = 0

  for i in range(0, len(test_en_list), batch_size):
    batch_en = test_en_list[i:i + batch_size]
    batch_de = test_de_list[i:i + batch_size]

    if len(batch_en) == 0:
      continue

    input_tensor, target_tensor = make_batch(batch_en, batch_de, len(batch_en))

    input_tensor = input_tensor.to(device)
    target_tensor = target_tensor.to(device)

    output = model(input_tensor, target_tensor)

    output_dim = output.shape[-1]
    output = output.reshape(-1, output_dim)
    target_tensor = target_tensor.reshape(-1)

    loss = criterion(output, target_tensor)
    total_loss += loss.item()

  avg_test_loss = total_loss / len(test_en_list)
  print(f'Test Loss: {avg_test_loss:.4f}')

Test Loss: 0.0878


In [24]:
def translate_sentence(sentence, encoder, decoder, en_token_dict, de_token_lookup, device, max_len=50):
    encoder.eval()
    decoder.eval()

    tokens = tokenize_sequence(sentence, en_token_dict)
    input_tensor = torch.LongTensor(tokens).unsqueeze(0).to(device)

    with torch.no_grad():
        encoder_outputs, (hidden, cell) = encoder(input_tensor)

    trg_indexes = [de_token_dict['<SOS>']]

    for i in range(max_len):
        trg_tensor = torch.LongTensor([trg_indexes[-1]]).to(device)

        with torch.no_grad():
            output, (hidden, cell) = decoder(trg_tensor, hidden, cell, encoder_outputs)

        pred_token = output.argmax(1).item()
        trg_indexes.append(pred_token)

        if pred_token == de_token_dict['<EOS>']:
            break

    trg_tokens = [de_token_lookup[idx] for idx in trg_indexes]

    return ' '.join(trg_tokens)

model.to(device)

test_sentence = "I want an orange."
translation = translate_sentence(test_sentence, encoder, decoder, en_token_dict, de_token_lookup, device)
print(f"English: {test_sentence}")
print(f"German: {translation}")

test_sentence_2 = "A loaf of bread, a container of milk, and a stick of butter."
translation_2 = translate_sentence(test_sentence_2, encoder, decoder, en_token_dict, de_token_lookup, device)
print(f"English: {test_sentence_2}")
print(f"German: {translation_2}")

English: I want an orange
German: <SOS> orangefarbenen orange , dass ein paar blumen sind , dass ein orangefarbenes orangefarbenen schild wird . es wird . es sieht . von oben zu sehen ist . es sieht . es sieht zu . . es werden . es schneit . auf einen orangefarbenen gegenstand . werden . es
English: A loaf of bread, a container of milk, and a stick of butter.
German: <SOS> ein etwa zehn , wo ein stück sich , einen kreis und einen stock auf dem boden . . ist . “ . ein stück himmel . . es . einen kreis . ist . ein stück himmel . . es sich . eine reihe von einem golden retriever .


The first text (put through google translate) means

*"Orange—that there are a few flowers, that an orange sign is taking shape. It is happening. It is visible. It can be seen from above. It looks on. It is coming to be. It is snowing—onto an orange object. It is becoming."*

and the second *"A space of about ten [feet], where a piece—a circle and a stick—lies on the ground... It is a piece of sky... It is a circle... It is a piece of sky... It is a Golden Retriever."*

The model has probably learned something but clearly it has room to be worked on.

## Part 3

In [32]:
class TwoHeadAttention(nn.Module):
  def __init__(self, model_dim):
    super().__init__()
    self.model_dim = model_dim
    head_dim = model_dim // 2

    self.attention1 = Attention(head_dim)
    self.attention2 = Attention(head_dim)

    self.W_q1 = nn.Linear(model_dim, head_dim)
    self.W_k1 = nn.Linear(model_dim, head_dim)
    self.W_v1 = nn.Linear(model_dim, head_dim)

    self.W_q2 = nn.Linear(model_dim, head_dim)
    self.W_k2 = nn.Linear(model_dim, head_dim)
    self.W_v2 = nn.Linear(model_dim, head_dim)

    self.W_out = nn.Linear(model_dim, model_dim)

  def forward(self, query_input, key_input=None, value_input=None):
    if key_input is None:
        key_input = query_input
    if value_input is None:
        value_input = query_input #handling flexibility for cross-attention and self attention

    query1 = self.W_q1(query_input)
    key1 = self.W_k1(key_input)
    value1 = self.W_v1(value_input)

    query2 = self.W_q2(query_input)
    key2 = self.W_k2(key_input)
    value2 = self.W_v2(value_input)

    attn_out1_context, _ = self.attention1(query1, key1, value1)
    attn_out2_context, _ = self.attention2(query2, key2, value2)

    combined_context = torch.cat((attn_out1_context, attn_out2_context), dim=-1)

    return self.W_out(combined_context)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:x.size(0), :]

with torch.no_grad():
  input_tensor = torch.randn(1, 10, 64)
  test_mha = TwoHeadAttention(64)
  print(test_mha(input_tensor).shape)

  encoder_output_for_cross_attn = torch.randn(1, 15, 64)
  decoder_input_for_cross_attn = torch.randn(1, 10, 64)
  cross_attn_output = test_mha(query_input=decoder_input_for_cross_attn,
                               key_input=encoder_output_for_cross_attn,
                               value_input=encoder_output_for_cross_attn)
  print(cross_attn_output.shape)

torch.Size([1, 10, 64])
torch.Size([1, 10, 64])


In [64]:
from torch import nn
import torch
import torch.nn.functional as F
import math

class tEncoder(nn.Module):
  def __init__(self, lexicon_size, emb_dim, model_dim, feed_forward_dim, max_seq_len=100):
    super().__init__()
    assert emb_dim == model_dim, "Embedding dimension must equal model dimension for Positional Encoding"

    self.embedding = nn.Embedding(lexicon_size, emb_dim, padding_idx=0)
    self.pos_encoder = PositionalEncoding(emb_dim, max_seq_len)

    self.attention = TwoHeadAttention(model_dim)
    self.attn_layer_norm = nn.LayerNorm(model_dim)

    self.ff_fc1 = nn.Linear(model_dim, feed_forward_dim)
    self.ff_fc2 = nn.Linear(feed_forward_dim, model_dim)
    self.ff_layer_norm = nn.LayerNorm(model_dim)

    self.dropout = nn.Dropout(0.1)

  def forward(self, input_tokens):
    embedded = self.embedding(input_tokens)
    embedded = self.dropout(embedded)

    embedded = embedded.permute(1, 0, 2)
    X = self.pos_encoder(embedded)
    X = self.dropout(X)
    X = X.permute(1, 0, 2)

    attn_output = self.attention(X)
    X = self.attn_layer_norm(X + self.dropout(attn_output))

    ff_output = self.ff_fc2(F.relu(self.ff_fc1(X)))
    X = self.ff_layer_norm(X + self.dropout(ff_output))

    return X

class tDecoder(nn.Module):
  def __init__(self, lexicon_size, emb_dim, model_dim, feed_forward_dim, max_seq_len=100, output_dim=None):
    super().__init__()

    self.embedding = nn.Embedding(lexicon_size, emb_dim, padding_idx=0)
    self.pos_encoder = PositionalEncoding(emb_dim, max_seq_len)

    self.self_attention = TwoHeadAttention(model_dim)
    self.self_attn_layer_norm = nn.LayerNorm(model_dim)

    self.cross_attention = TwoHeadAttention(model_dim)
    self.cross_attn_layer_norm = nn.LayerNorm(model_dim)

    self.ff_fc1 = nn.Linear(model_dim, feed_forward_dim)
    self.ff_fc2 = nn.Linear(feed_forward_dim, model_dim)
    self.ff_layer_norm = nn.LayerNorm(model_dim)

    self.fc_out = nn.Linear(model_dim, output_dim)

    self.dropout = nn.Dropout(0.1)

  def forward(self, target_tokens, encoder_output):
    embedded = self.embedding(target_tokens)
    embedded = self.dropout(embedded)

    embedded = embedded.permute(1, 0, 2)
    X = self.pos_encoder(embedded)
    X = self.dropout(X)
    X = X.permute(1, 0, 2)

    self_attn_output = self.self_attention(X)
    X = self.self_attn_layer_norm(X + self.dropout(self_attn_output))

    cross_attn_output = self.cross_attention(query_input=X, key_input=encoder_output, value_input=encoder_output)
    X = self.cross_attn_layer_norm(X + self.dropout(cross_attn_output))

    ff_output = self.ff_fc2(F.relu(self.ff_fc1(X)))
    X = self.ff_layer_norm(X + self.dropout(ff_output))

    output = self.fc_out(X)

    return output

class Transformer(nn.Module):
  def __init__(self):
    super().__init__()
    self.encoder_layer1 = tEncoder(len(en_token_dict), 64, 64, 128)
    self.encoder_layer2 = tEncoder(len(en_token_dict), 64, 64, 128)

    self.decoder_layer1 = tDecoder(len(de_token_dict), 64, 64, 128, output_dim=len(de_token_dict))
    self.decoder_layer2 = tDecoder(len(de_token_dict), 64, 64, 128, output_dim=len(de_token_dict))

    self.fc = nn.Linear(64, len(de_token_dict))

  def forward(self, src_tokens, trg_tokens):
    encoder_output = self.encoder_layer1(src_tokens)
    encoder_output = self.encoder_layer2(encoder_output)

    decoder_output = self.decoder_layer1(trg_tokens, encoder_output)
    decoder_output = self.decoder_layer2(trg_tokens, encoder_output)

    decoder_output = decoder_output.permute(1, 0, 2)
    decoder_output = self.fc(decoder_output)

    return decoder_output


In [65]:
#sanity check

test_transformer = Transformer()

test_transformer.eval()

dummy_data = train_data[0:16]

print(dummy_data.head())

dummy_batch = make_batch(dummy_data['token_en'].to_list(), dummy_data['token_de'].to_list(), 16)

print(dummy_batch[0].shape)
print(dummy_batch[1].shape)

inference = test_transformer(dummy_batch[0], dummy_batch[1])

print("model inference shape:", inference.shape)

                                                token_en  \
9363   [9795, 8120, 4965, 10269, 4660, 7278, 7082, 84...   
23806  [9795, 4115, 20, 118, 9795, 340, 413, 283, 873...   
5653   [2657, 5429, 10269, 9898, 4670, 9850, 10269, 9...   
16613  [10269, 6059, 7393, 211, 7183, 73, 2919, 2433,...   
10512  [9795, 1576, 2616, 1258, 220, 1411, 6059, 5278...   

                                                token_de  
9363   [6042, 4860, 7338, 8482, 16932, 6042, 1766, 98...  
23806  [6042, 15819, 12281, 18370, 6042, 8243, 1578, ...  
5653   [1988, 8960, 18370, 14534, 9928, 18370, 3861, ...  
16613  [12102, 13725, 17651, 6626, 10743, 14780, 1958...  
10512  [11924, 10230, 13938, 1654, 2669, 9003, 17905,...  
torch.Size([16, 36])
torch.Size([16, 26])


RuntimeError: Expected tensor for argument #1 'indices' to have one of the following scalar types: Long, Int; but got torch.FloatTensor instead (while checking arguments for embedding)

Here is my prototype of a sequence to sequence transformer. The implementation's forward pass does not work but the attention head, encoder, decoder, and sinusoidal position encoder are implemented. For time reasons, I'm unable to complete the implementation or train.